# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the **Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution** dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

This notebook will: 
- Load and inspect the metadata
- Review available record sets and fields (referenced by `@id`)
- Extract tabular data for analysis and visualization
- Demonstrate basic exploratory data analysis (EDA) and visualization

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

# Print some keywords
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available Record Sets, Fields, and their `@id`.

We will use the `dataset.record_sets` property to enumerate all record sets and their fields, referencing each by its unique `@id`. This allows accurate loading and flexible manipulation.

In [ ]:
# List record sets and their fields by @id
for record_set in dataset.record_sets:
    print(f"Record Set: {record_set['@id']} -- {record_set.get('name','')}")
    if 'field' in record_set:
        print('  Fields:')
        for field in record_set['field']:
            field_id = field['@id']
            field_name = field.get('name', '')
            field_type = field.get('dataType', '')
            print(f"    - {field_id} (name: {field_name}, type: {field_type})")
    print('---')

# Print available record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print('Available record sets (@id):', record_set_ids)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Each record set and field is referenced by its `@id`. Below we demonstrate extraction for all available record sets.

In [ ]:
# Extract data from all record sets
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded records for record set @{record_set_id}: {df.shape[0]} rows")
    print(f"Columns (@id): {df.columns.tolist()}")
    print("-- Example rows --")
    print(df.head())
    print('============================')

# For convenience, select the first record set for demo
if len(record_set_ids) > 0:
    demo_record_set_id = record_set_ids[0]
    demo_df = dataframes[demo_record_set_id]
    print(f"Preview of first record set @{demo_record_set_id}")
    print(demo_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps including filtering, normalization, and grouping.

For demonstration, choose a numeric field and a group field using their `@id`. Refer to the data overview output above to select valid field IDs.

In [ ]:
# Set up demo field IDs based on overview (change as needed)
numeric_field_id = None
group_field_id = None
# Detect a numeric field by type or sample value
from numpy import number
if len(record_set_ids) > 0:
    record_set_id = demo_record_set_id
    df = dataframes[record_set_id]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            group_field_id = col
            break

if numeric_field_id:
    print(f'Numeric field selected (@id): {numeric_field_id}')
    threshold = df[numeric_field_id].mean() # Use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field_id if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please refer to previous data overview to select a numeric field @id for your dataset.")

## 5. Visualization
Visualize distributions or relationships between fields. For example, plot the normalized numeric field and group sizes.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 5))
    # Histogram of normalized numeric field
    filtered_df[norm_col].plot.hist(bins=10, alpha=0.7, color='steelblue')
    plt.title(f"Histogram of normalized {numeric_field_id}")
    plt.xlabel(norm_col)
    plt.tight_layout()
    plt.show()

    # Bar plot of group sizes
    group_counts = filtered_df[group_field_id].value_counts().head(10)
    group_counts.plot.bar()
    plt.title(f"Top 10 groups by {group_field_id} size")
    plt.xlabel(group_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Visualization skipped: suitable numeric and group field not found. Please update field @id variables.")

## 6. Conclusion
This notebook provided an overview and exploration of the FAIR^2 dataset on clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors using the `mlcroissant` library.

- All entities (record sets, fields, columns) were referenced by their unique `@id`, ensuring reproducibility and schema compliance.
- We loaded metadata, reviewed available record sets and fields, and extracted tabular data for analysis.
- Basic EDA operations and visualizations demonstrated how to filter, normalize, and group clinical records.

For deeper insights, users may:
- Explore additional record sets
- Customize the analysis based on specific clinical or molecular attributes (using their `@id`)
- Integrate `mlcroissant` with machine learning pipelines

Refer to the [Croissant JSON-LD schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) for entity IDs and extended metadata.